In [114]:
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env
import numpy as np

In [115]:
class GridMaps(gym.Env):
    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Dict({
            'agent': spaces.Box(low=0, high=5, shape=(2, ), dtype=np.int32),
            'goal': spaces.Box(low=0, high=5, shape=(2, ), dtype=np.int32),
            'map': spaces.Box(low=0, high=1, shape=(5, 5), dtype=np.int32)
        })
        # haut   0
        # bas    1
        # gauche 2
        # droite 3
        self.action_space = spaces.Discrete(4)
        self.steps = 0
        self.position = np.array([0, 0], dtype=np.int32)
        self.goals = np.array([4, 3], dtype=np.int32)
        self.max_step = 20

    def reset(self, *, seed = None, options = None):
        super().reset(seed=seed, options=options)
        if options and 'map' in options:
            self.map = options['map']
        else:
            self.map = self.np_random.integers(low=0, high=2, size=(5, 5), dtype=np.int32)
        if options and 'position' in options:
            self.position = options['position']
        else:
            self.position = self.np_random.integers(0, 5, size=(2, ),dtype=np.int32)
        if options and 'goals' in options:
            self.goals = options['goals']
        else:
            self.goals = self.np_random.integers(0, 5, size=(2, ),dtype=np.int32)
            while self.map[self.goals[0], self.goals[1]] == 1:
                self.goals = self.np_random.integers(0, 5, size=(2, ),dtype=np.int32)
        self.map[self.position[0]][self.position[1]] = 0
        self.steps = 0
        return {'agent': self.position.copy(),'goal': self.goals.copy(),'map': self.map.copy()}, {}

    def step(self, action):
        self.steps += 1
        reward = 0
        x, y = self.position
        if action == 0:
            y -= 1
        if action == 1:
            y += 1
        if action == 2:
            x -= 1
        if action == 3:
            x += 1

        if x< 0 or x> 4 or y < 0 or y > 4 or self.map[x][y] == 1:
            reward -= 5
        else:
            self.position = np.array([x, y])
        terminated = np.array_equal(self.goals, self.position)
        truncated = self.steps == self.max_step
        if terminated:
            reward = 10
        if truncated:
            reward = -10
        return {'agent': self.position,'goal': self.goals,'map': self.map}, reward, terminated, truncated, {}

In [116]:
env = GridMaps()

obs, info = env.reset()

for i in range(2):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action=action)
    print(obs)
    print('='*20)

check_env(env=env)

{'agent': array([1, 1], dtype=int32), 'goal': array([4, 0], dtype=int32), 'map': array([[1, 0, 1, 1, 0],
       [0, 0, 0, 1, 0],
       [1, 1, 0, 0, 1],
       [1, 1, 0, 1, 1],
       [0, 1, 0, 0, 1]], dtype=int32)}
{'agent': array([1, 0], dtype=int32), 'goal': array([4, 0], dtype=int32), 'map': array([[1, 0, 1, 1, 0],
       [0, 0, 0, 1, 0],
       [1, 1, 0, 0, 1],
       [1, 1, 0, 1, 1],
       [0, 1, 0, 0, 1]], dtype=int32)}


In [117]:
from stable_baselines3 import PPO

In [118]:
agent = PPO('MultiInputPolicy', env=env, device='cuda')

agent.learn(total_timesteps=100_000)

In [119]:
act = {
    'haut': 0,
    'bas': 1,
    'gauche': 2,
    'droite': 3
}

def get_action(number):
    for k,v in act.items():
        if v == number:
            return k, v

In [120]:
obs, info = env.reset(options={
    'goals':[3, 3]
})
a = obs['agent']
here = obs['map'].copy()
here[a[0]][a[1]] = 5
print(here.T)
while True:
    action, _ = agent.predict(obs)
    obs, reward, terminated, truncated, info = env.step(action=action)
    a = obs['agent']
    here = obs['map'].copy()
    here[a[0]][a[1]] = 5
    print(get_action(action))
    print(here.T)
    if terminated:
        print('Jeu Termine',terminated)
        break
    elif truncated:
        print('Max step atteinte', truncated)
        break

[[1 5 1 0 0]
 [1 0 0 0 1]
 [0 0 0 0 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('droite', 3)
[[1 5 1 0 0]
 [1 0 0 0 1]
 [0 0 0 0 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('bas', 1)
[[1 0 1 0 0]
 [1 5 0 0 1]
 [0 0 0 0 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('bas', 1)
[[1 0 1 0 0]
 [1 0 0 0 1]
 [0 5 0 0 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('bas', 1)
[[1 0 1 0 0]
 [1 0 0 0 1]
 [0 5 0 0 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('droite', 3)
[[1 0 1 0 0]
 [1 0 0 0 1]
 [0 0 5 0 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('bas', 1)
[[1 0 1 0 0]
 [1 0 0 0 1]
 [0 0 5 0 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('droite', 3)
[[1 0 1 0 0]
 [1 0 0 0 1]
 [0 0 0 5 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('bas', 1)
[[1 0 1 0 0]
 [1 0 0 0 1]
 [0 0 0 5 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('bas', 1)
[[1 0 1 0 0]
 [1 0 0 0 1]
 [0 0 0 5 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('bas', 1)
[[1 0 1 0 0]
 [1 0 0 0 1]
 [0 0 0 5 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('gauche', 2)
[[1 0 1 0 0]
 [1 0 0 0 1]
 [0 0 5 0 0]
 [1 1 1 1 0]
 [1 0 1 0 1]]
('bas', 1)
[[1 0 1 0 0]
 [1 0 0 0 1]
 [0 0 5 0 0]
 [1 1 1 1 0]
 [1 0 1 0 1]

In [129]:
import pygame

pygame.init()

SIZE = 500
WIDTH = 500 // 5
WHITE_COLOR = (255, 255, 255)

screen = pygame.display.set_mode((SIZE + 5, SIZE + 5))

FPS = 5
time = pygame.time.Clock()

loop = True

obs, info = env.reset()
goal = obs['goal']
def create_grid(obs):
    m = obs['map']
    for i in range(len(m[0])):
        for j in range(len(m[1])):
            if m[i, j] == 0:
                pygame.draw.rect(screen,pygame.Color(WHITE_COLOR), pygame.Rect(i * WIDTH + 5, j * WIDTH +5 , WIDTH - 5, WIDTH - 5))
            else:
                pygame.draw.rect(screen,pygame.Color(0, 0, 0), pygame.Rect(i * WIDTH + 5, j * WIDTH +5 , WIDTH - 5, WIDTH - 5))

def draw_obj(x, y, color, decalage = 0):
    pygame.draw.circle(screen,pygame.Color(color), (x * WIDTH + WIDTH // 2 - decalage, y * WIDTH + WIDTH // 2), 20)

terminated = False
decalage = 0
while loop:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            loop = False
    screen.fill(color=WHITE_COLOR)
    create_grid(obs)
    if terminated:
        decalage = 10
    draw_obj(obs['agent'][0],obs['agent'][1], (0, 0, 255), decalage=decalage)
    draw_obj(goal[0], goal[1], (200,0, 100), decalage=-decalage)
    pygame.display.update()
    time.tick(FPS)
    if terminated == False:
        action, info = agent.predict(obs)
        obs, reward, terminated, truncated, info = env.step(action=action)
pygame.quit()